In [1]:
%cd /content/drive/MyDrive/bank-churn-classification

/content/drive/MyDrive/bank-churn-classification


# Importing Libraries

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import textwrap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, MinMaxScaler, StandardScaler

plt.style.use('seaborn-v0_8-darkgrid')

# Loading Dataset

In [3]:
df = pd.read_csv('Data.csv', index_col='id')
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
id,,,,,,,,,,,,,
0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


# Exploratory Data Analysis

## Basic Data Exploration

In [4]:
# Checking dataset shape
df.shape

(165034, 13)

In [5]:
# Checking column types
df.dtypes

CustomerId           int64
Surname             object
CreditScore          int64
Geography           object
Gender              object
Age                float64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard          float64
IsActiveMember     float64
EstimatedSalary    float64
Exited               int64
dtype: object

In [6]:
# Stat of numerical values
df.describe()

,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,1.650340e+05,165034.000000,165034.000000,165034.000000,165034.000000,165034.000000,165034.000000,165034.000000,165034.000000,165034.000000
mean,1.569201e+07,656.454373,38.125888,5.020353,55478.086689,1.554455,0.753954,0.497770,112574.822734,0.211599
std,7.139782e+04,80.103340,8.867205,2.806159,62817.663278,0.547154,0.430707,0.499997,50292.865585,0.408443
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,11.580000,0.000000
25%,1.563314e+07,597.000000,32.000000,3.000000,0.000000,1.000000,1.000000,0.000000,74637.570000,0.000000
50%,1.569017e+07,659.000000,37.000000,5.000000,0.000000,2.000000,1.000000,0.000000,117948.000000,0.000000
75%,1.575682e+07,710.000000,42.000000,7.000000,119939.517500,2.000000,1.000000,1.000000,155152.467500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.000000,1.000000,199992.480000,1.000000


In [7]:
# Checking number of null values
df.isna().sum()

CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [8]:
# Total duplicate rows
df.duplicated(subset=['CustomerId']).sum()

141813

In [9]:
# Making lists of numerical and categorical columns
numerical_cols = [cname for cname in df.columns if df[cname].dtype in ['int64', 'float64']]
categorical_cols = [cname for cname in df.columns if df[cname].dtype == "object"]

# Printing numerical and categorical column lists
print('Numerical Columns: ', numerical_cols)
print('Categorical Columns: ', categorical_cols)

Numerical Columns:  ['CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
Categorical Columns:  ['Surname', 'Geography', 'Gender']


In [10]:
# Calculate number of unique values and unique values for each categorical features
unique_counts = df[categorical_cols].nunique()
unique_values = df[categorical_cols].apply(lambda x: x.unique())

# Create new dataframe with the results
pd.DataFrame({'Number of Unique Values': unique_counts, 'Unique Values': unique_values})

,Number of Unique Values,Unique Values
Surname,2797,"[Okwudilichukwu, Okwudiliolisa, Hsueh, Kao, Ch..."
Geography,3,"[France, Spain, Germany]"
Gender,2,"[Male, Female]"


In [11]:
# Surname value counts
df['Surname'].value_counts()

Hsia         2456
T'ien        2282
Hs?          1611
Kao          1577
Maclean      1577
             ... 
Samaniego       1
Lawley          1
Bonwick         1
Tennant         1
Elkins          1
Name: Surname, Length: 2797, dtype: int64

In [12]:
# Geography value counts
df['Geography'].value_counts()

France     94215
Spain      36213
Germany    34606
Name: Geography, dtype: int64

In [13]:
# Geography value counts
df['Gender'].value_counts()

Male      93150
Female    71884
Name: Gender, dtype: int64

In [14]:
# Exited value counts
df['Exited'].value_counts()

0    130113
1     34921
Name: Exited, dtype: int64

## Data Viz

In [15]:
# Create countplot for the Gender
histogram1 = go.Histogram(x=df['Gender'], name='Gender')

# Create countplot for the Geography
histogram2 = go.Histogram(x=df['Geography'], name='Geography')

# Create subplots
fig = make_subplots(rows=1, cols=2)

# Add countplot for the Gender to subplot
fig.add_trace(histogram1, row=1, col=1)

# Add countplot for the Geography to subplot
fig.add_trace(histogram2, row=1, col=2)

# Update layout
fig.update_layout(title='Countplots')

# Show plot
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [16]:
# Create subplots with Plotly Express
fig = px.histogram(df, x='Gender', color='Exited', facet_col='Geography')

# Update layout
fig.update_layout(title='Countplots of Gender and Geography with Exited',
                  xaxis=dict(title='Gender'), yaxis=dict(title='Count'))

# Show plot
fig.show()

In [17]:
df_hist = df[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']]

fig = make_subplots(rows=2, cols=4, subplot_titles=df_hist.columns)

fig.add_trace(go.Histogram(x=df['CreditScore'], name='CreditScore'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Age'], name='Age'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['Tenure'], name='Tenure'), row=1, col=3)
fig.add_trace(go.Histogram(x=df['Balance'], name='Balance'), row=1, col=4)
fig.add_trace(go.Histogram(x=df['NumOfProducts'], name='NumOfProducts'), row=2, col=1)
fig.add_trace(go.Histogram(x=df['HasCrCard'], name='HasCrCard'), row=2, col=2)
fig.add_trace(go.Histogram(x=df['IsActiveMember'], name='IsActiveMember'), row=2, col=3)
fig.add_trace(go.Histogram(x=df['EstimatedSalary'], name='EstimatedSalary'), row=2, col=4)

# Update layout
fig.update_layout(title='Individual Displots of Eight Columns', showlegend=False)
fig.update_xaxes(title_text='Value', row=2, col=1)
fig.update_yaxes(title_text='Probability Density', row=1, col=1)

# Show plot
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [27]:
df_hist = df[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'IsActiveMember', 'EstimatedSalary']]

fig = make_subplots(rows=2, cols=4, subplot_titles=df_hist.columns)

fig.add_trace(go.Box(x=df['CreditScore'], name='CreditScore'), row=1, col=1)
fig.add_trace(go.Box(x=df['Age'], name='Age'), row=1, col=2)
fig.add_trace(go.Box(x=df['Tenure'], name='Tenure'), row=1, col=3)
fig.add_trace(go.Box(x=df['Balance'], name='Balance'), row=1, col=4)
fig.add_trace(go.Box(x=df['NumOfProducts'], name='NumOfProducts'), row=2, col=1)
fig.add_trace(go.Box(x=df['IsActiveMember'], name='IsActiveMember'), row=2, col=2)
fig.add_trace(go.Box(x=df['EstimatedSalary'], name='EstimatedSalary'), row=2, col=3)

# Update layout
fig.update_layout(title='Individual Box Plots of Eight Columns', showlegend=False)
fig.update_xaxes(title_text='Feature', row=2, col=1)
fig.update_yaxes(title_text='Value', row=1, col=1)

# Rotate y-axis labels
fig.update_yaxes(tickangle=90)

# Show plot
fig.show()


Output hidden; open in https://colab.research.google.com to view.